# Absolute reconstruction energy errors

Compute MAE, P10, P90, maximum and tail counts for Uniform, Model and Adaptive, with and without closure constraints. Retain the explicit common energy mask and compute per-atom absolute errors from the same structures.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'provenance/hpc_download_plan.tsv').is_file())
for package in reversed(('McUtils', 'Psience', 'ocen')):
    sys.path.insert(0, str(ROOT / 'software' / package))
print(ROOT)


In [ ]:
import csv
import numpy as np
path = ROOT / 'data/remote/entropy/geom-constrained-comparison-20261121/results/merged/comparison.npz'
records = []
with np.load(path, allow_pickle=False) as data:
    mask = data['energy_common_mask'].astype(bool)
    assert mask.sum() == 99488
    for condition in ('unconstrained', 'constrained'):
        values = data[condition + '_delta_energy_kjmol']
        for e, encoder in enumerate(data['encoder_order'].astype(str)):
            for p, potential in enumerate(data['potential_order'].astype(str)):
                for per_atom in (False, True):
                    # Absolute errors, with per-atom normalization before summarizing.
                    signed = values[mask, p, e]
                    if per_atom:
                        signed = signed / data['atom_counts'][mask]
                    absolute = np.abs(signed)
                    assert np.all(np.isfinite(absolute))
                    p10, p90 = np.quantile(absolute, [.1, .9])
                    cutoff = .15 if per_atom else 10.
                    records.append(dict(condition=condition, encoder=encoder, potential=potential,
                        units='kJ/mol/atom' if per_atom else 'kJ/mol', n=len(absolute),
                        MAE=absolute.mean(), P10=p10, P90=p90, maximum=absolute.max(),
                        cutoff=cutoff, count_above=int(np.count_nonzero(absolute > cutoff))))
output = ROOT / 'outputs/statistics'; output.mkdir(parents=True, exist_ok=True)
with (output / 'absolute_energy_errors.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(records[0]))
    writer.writeheader(); writer.writerows(records)
import pandas as pd
pd.DataFrame(records)
